# **Random Meal Planner**

This notebook tests the `RandomMealPlanner`, which generates a meal plan by randomly selecting recipes from the dataset. This serves as a baseline for comparing the performance of other meal planners.

In [1]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random, seed

from engines import (
    RandomMealPlanner,
    filter_and_add_recipes,
    get_pantry_ingredient,
    load_all_ingredients,
    make_preferences,
    parse_recipes,
)
from models import (
    MealPlanningEnvironment,
    Pantry,
)


## Environment Setup

In [2]:
seed(1)

In [3]:
preferences = make_preferences()

In [4]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [5]:
CURRENT_DATE = datetime.now()

In [6]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [7]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [8]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [9]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [10]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-05-26
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-29
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-31
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Saturated Fat: 0.0 g
		Fiber: 2.2 g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---


In [11]:
with open("../recipe_extraction/supplemented_structured_recipes.json", "r") as file:
    all_recipes = json.load(file)

In [12]:
all_recipes = parse_recipes(all_recipes)

In [13]:
NUM_FILTERED_RECIPES = 100
NUM_EXTRA_RECIPES = 50

In [14]:
final_recipes = filter_and_add_recipes(
    all_recipes, [ingredient.name for ingredient in pantry_ingredients], NUM_FILTERED_RECIPES, NUM_EXTRA_RECIPES
)

In [15]:
print(f"Number of recipes before filtering: {len(all_recipes)}")
print(f"Number of recipes after filtering: {len(final_recipes)}")

Number of recipes before filtering: 19716
Number of recipes after filtering: 150


In [16]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=final_recipes,
    pantry=pantry,
    preferences=preferences,
)

In [17]:
all_ingredient_names = []

for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

In [18]:
all_ingredient_costs = {}

for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = random()

In [19]:
meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

## Running the Random Meal Planner

In [20]:
planner = RandomMealPlanner(meal_planning_environment, random_seed=1)

In [21]:
best_meal_plan = planner.generate_meal_plan()

### *Random Meal Plan Results*

In [22]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0.000000,800.000000,1d
1,broccoli,1500,255.866667,1244.133333,4d
2,rice,1000,612.783333,387.216667,6d


In [23]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Meal 1,Meal 2,Meal 3
Day 1,Moroccan Chicken with Kumquats and Prunes,"Open-Face Prosciutto, Fresh Ricotta, and Red-O...",Country Captain Chicken
Day 2,Glazed Japanese Chicken Meatballs on Skewers,"Shrimp, Chicken, and Andouille Gumbo","Steamed Red Snapper with Ginger, Chiles, and S..."
Day 3,Sesame Miso Vinaigrette,Tarragon Tartar Sauce,Penne with Vegetables and Olives
Day 4,Star Fish,Farmers' Market Quinoa Salad,"Melon, Basil, and Feta Salad With Balsamic–Red..."
Day 5,"Fried Rice with Ham, Egg, and Scallions",Creamed Broccoli with Parmesan,Chocolate Chip Apricot Bars
Day 6,Grilled Stuffed Peppers,Riesling-Braised Sauerkraut and Apples,"Steamed Broccoli with Olive Oil, Garlic, and L..."
Day 7,Asian Bow-Tie Pasta Salad with Shrimp and Vege...,"Broccoli, Red Pepper, and Cheddar Chowder",Spicy Black Beans with Bell Peppers and Rice


In [24]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 163
Total cost: €85.21


,Ingredient,Quantity to Buy (g),Cost (€)
0,Asian sesame oil,1.5,0.01
1,Cayenne pepper,1.8,0.00
2,Dijon mustard,3.3,0.03
3,Fresh rosemary,0.7,0.00
4,Granny Smith apple,5.9,0.03
...,...,...,...
159,white miso,9.9,0.07
160,white wine,118.3,0.06
161,white wine vinegar,9.9,0.05
162,zucchini,3.2,0.02


In [25]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,7839.0 kcal,532.2 g,5839.0 kcal,482.2 g
Day 2,3211.1 kcal,276.7 g,1211.1 kcal,226.7 g
Day 3,1560.6 kcal,50.5 g,-439.4 kcal,0.5 g
Day 4,2954.8 kcal,225.5 g,954.8 kcal,175.5 g
Day 5,819.4 kcal,19.6 g,-1180.6 kcal,-30.4 g
Day 6,3213.4 kcal,136.8 g,1213.4 kcal,86.8 g
Day 7,1334.0 kcal,61.8 g,-666.0 kcal,11.8 g
